In [ ]:
import shutil
from pathlib import Path

from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import HfApi, hf_hub_download

load_dotenv()

REPO_ID = "Savassan/mnemo"
PROJECT_ROOT = Path("..").resolve()
PROJECTORS_DIR = PROJECT_ROOT / "data" / "alpha_edit_projectors"
LAYER_STATS_DIR = PROJECT_ROOT / "data" / "layer_stats"
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets"
REMOTE_BASE = "artifacts/qwen2.5-7b"


def upload_artifacts() -> None:
    api = HfApi()

    projector_file = PROJECTORS_DIR / "null_space_project.pt"
    if not projector_file.exists():
        raise FileNotFoundError(projector_file)

    files_to_upload = [projector_file]
    files_to_upload.extend(sorted(LAYER_STATS_DIR.rglob("*.npz")))

    for local_file in files_to_upload:
        relative_path = local_file.relative_to(PROJECT_ROOT).as_posix()
        remote_path = f"{REMOTE_BASE}/{relative_path}"
        api.upload_file(
            path_or_fileobj=str(local_file),
            path_in_repo=remote_path,
            repo_id=REPO_ID,
            repo_type="dataset",
        )
        print(f"Uploaded {relative_path}")


def download_artifacts(overwrite: bool = False) -> None:
    api = HfApi()
    remote_files = [
        path
        for path in api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
        if path.startswith(f"{REMOTE_BASE}/data/")
    ]

    if not remote_files:
        raise FileNotFoundError(f"No artifact files found under {REPO_ID}/{REMOTE_BASE}")

    for remote_file in remote_files:
        relative_path = remote_file.removeprefix(f"{REMOTE_BASE}/")
        destination = PROJECT_ROOT / Path(relative_path)
        destination.parent.mkdir(parents=True, exist_ok=True)

        if destination.exists() and not overwrite:
            print(f"Skipped {relative_path}")
            continue

        cached_file = hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=remote_file,
            force_download=overwrite,
        )
        shutil.copy2(cached_file, destination)
        print(f"Downloaded {relative_path}")


def download_dataset(dataset_name: str, overwrite: bool = False) -> None:
    destination = DATASETS_DIR / dataset_name
    if destination.exists() and not overwrite:
        print(f"Skipped data/datasets/{dataset_name}")
        return

    if destination.exists():
        shutil.rmtree(destination)

    dataset = load_dataset(REPO_ID, name=dataset_name)
    dataset.save_to_disk(str(destination))
    print(f"Saved data/datasets/{dataset_name}")


def download_all_datasets(overwrite: bool = False) -> None:
    for dataset_name in ["cfact", "mquake"]:
        download_dataset(dataset_name, overwrite=overwrite)

In [ ]:
# Upload AlphaEdit/MEMIT artifact files from your local machine:
# upload_artifacts()

# Download AlphaEdit/MEMIT artifact files on the rented machine:
# download_artifacts()

# Download benchmark datasets into data/datasets/:
# download_all_datasets()

# Or download only one dataset:
# download_dataset("cfact")
# download_dataset("mquake")

# Re-download and replace existing local copies if needed:
# download_artifacts(overwrite=True)
# download_all_datasets(overwrite=True)